# TFMN2 and TFMN3 Processing

## Overview

**Before sequencing**:
1. ~~From extracted robotic OD data, create table of TransferSamples.~~
2.  ~~From Nidhi and, get table with selected seqsamples. Curate. Verify correctness (can't have same seqsample in two different wells, can't have two samples in one well, all wells must exist - must start at A1 and be filled consecutively). Fix mistakes after discussion if necessary.~~
3.  ~~Merge two tables (seqsamples).~~
4.  ~~Create LIMS seqorder entry (and note if unrelated seqsamples are included), upload seqsamples to LIMS seqsamples table, and point seqsamples to seqorder (Google LIMS)  - delete table~~
5.  ~~create seqsamples plasmidsaurus name list  - delete table~~

**After sequencing**:
1. Create SeqOrder, Libraries, and SeqSamples. Verify seqsamples against LIMS.
2. From seqsamples, make LIMS measurements entries (not all seqsamples may be part of this experiment, so allow for other peoples samples) (all remaining seqsamples are measurements, but not all measurements are seqsamples) - delete table
3. Run analyses on seqsamples and upload results to LIMS, and link to them in measurements table



## Set-Up

In [ ]:
## Make sure running in aisynbio_env

In [2]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [4]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## Sync Google LIMS with DB and get number of expected samples

In [4]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [16]:
# Run a manual LIMS mirror db sync

from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2026-03-05 03:25:46,958 - lims_sync - INFO - Starting sync operation
2026-03-05 03:25:46,959 - lims_sync - INFO - Connecting to Google Sheets
2026-03-05 03:25:47,541 - lims_sync - INFO - Connecting to database
2026-03-05 03:25:47,775 - lims_sync - INFO - Found 17 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, Wells, Seqorders, Seqsamples, robotic_mt_samples, dgoA_alleles_new, dgoA_alleles_old, Strain_stocks_ANL, Plasmid_stocks_ANL
2026-03-05 03:25:47,776 - lims_sync - INFO - Syncing worksheet: Experiments
2026-03-05 03:25:48,761 - lims_sync - INFO - Retrieved 11 rows from Experiments
2026-03-05 03:25:48,841 - lims_sync - INFO - Inserted 1, updated 0 rows in Experiments
2026-03-05 03:25:48,853 - lims_sync - INFO - Marked 1 rows as deleted in Experiments
2026-03-05 03:25:53,854 - lims_sync - INFO - Syncing worksheet: Strains
2026-03-05 03:25:54,830 - lims_sync - INFO - Retrieved 1020 rows from Strains
2026-03-05 03:

{'start_time': '2026-03-05T03:25:46.958269',
 'end_time': '2026-03-05T03:29:25.023626',
 'success': True,
 'tables_synced': 17,
 'total_rows_inserted': 554,
 'total_rows_updated': 0,
 'total_rows_deleted': 1166,
 'errors': []}

In [ ]:
# LIMS_samples = query_lims(
#     'robotic_mt_samples',
#     filters={'Experiment': ['TFMN2']}
# )
# n_samples = len(LIMS_samples)

In [5]:
LIMS_wells = query_lims('Wells')
LIMS_wells_byRow = LIMS_wells.sort_values(['Row', 'Column'])['Well']
LIMS_wells_byCol = LIMS_wells.sort_values(['Column', 'Row'])['Well']

## Get TransferSamples

In [7]:
import pandas as pd

paths_to_OD_data = [
    '/storage/synbio/ai_synbio_data/experimental_data/robotic_od_data/TFMN2_01-14-26/TFMN2_robotic_OD_processed_final.csv',
    '/storage/synbio/ai_synbio_data/experimental_data/robotic_od_data/TFMN3_02-04-26/TFMN3_robotic_OD_processed_final.csv'
]
path_to_Nidhi_table = '/storage/nspahr/tmp/TFMN2 and TFMN3 sequencing plate.csv'

OD_df = pd.DataFrame()
for path in paths_to_OD_data:
    df_to_add = pd.read_csv(path).convert_dtypes().rename(columns={'Name':'sample_name'})
    OD_df = pd.concat([OD_df, df_to_add])

Nidhi_all_df = pd.read_csv(path_to_Nidhi_table).convert_dtypes()

expected_cols = ['Sequencing sample',
                 'Experiment',
                 'Sequencing plate',
                 'Sequencing plate well',
                 'Population or Single colony?',
                 'Robotic run plate',
                 'Robotic run plate well']

[print(x) for x in expected_cols if x not in Nidhi_all_df.columns]

expected_exp = ['TFMN2', 'TFMN3']
[print(x) for x in expected_exp if x not in Nidhi_all_df.Experiment.unique()]

Nidhi_df = Nidhi_all_df.loc[Nidhi_all_df['Experiment'].isin(expected_exp)]
Nidhi_df = Nidhi_df.sort_values(['Experiment', 'Sequencing plate','Sequencing plate well'])

other_df = Nidhi_all_df.loc[~Nidhi_all_df['Experiment'].isin(expected_exp)]

In [8]:
OD_df.columns

Index(['filename', 'experiment', 'file_ID', 'timestamp', 'series',
       'plate_index', 'transfer', 'reading', 'row', 'column', 'od', 'well',
       'measurement_type', 'culture_container', 'plate_type', 'start_date',
       'file_basename', 'bmg filename', 'datetime', 'resource id',
       'sample_name', 'Experiment', 'Type', 'Condition', 'strain',
       'Transforming DNA', 'Protocol', 'Parent sample', 'Replicate samples',
       'Plate name', 'Microtiter plate well', 'Unnamed: 11', 'background',
       'innoculation_timestamp', 'timepoint', 'trans_DNA concentration',
       'trans_DNA+conc'],
      dtype='object')

In [9]:
OD_df.head()

,filename,experiment,file_ID,timestamp,series,plate_index,transfer,reading,row,column,...,Parent sample,Replicate samples,Plate name,Microtiter plate well,Unnamed: 11,background,innoculation_timestamp,timepoint,trans_DNA concentration,trans_DNA+conc
0,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,0,...,ANLstock.ACN3575.colony1,"TFMN2.ACN3575.Iva.noDNA.1, TFMN2.ACN3575.Iva.n...",exp1,A1,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
1,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,1,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb.1, TFMN2.ACN3500.Iva....",exp1,A2,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
2,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,2,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb-DELvanK.1, TFMN2.ACN3...",exp1,A3,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
3,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,3,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.1, TF...",exp1,A4,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
4,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,4,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.gDNA3560.1, TFMN2.ACN3500.Iv...",exp1,A5,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>


In [10]:
Nidhi_df

,Experiment,Sequencing plate,Sequencing plate well,Robotic run plate,Robotic run plate well,Population or Single colony?,Sequencing sample
0,TFMN2,1,A1,1,B2,P,<NA>
9,TFMN2,1,A10,1,C2,P,<NA>
10,TFMN2,1,A11,1,C3,P,<NA>
11,TFMN2,1,A12,1,C4,P,<NA>
1,TFMN2,1,A2,1,B3,P,<NA>
...,...,...,...,...,...,...,...
247,TFMN3,3,E8,8,C3,P,<NA>
248,TFMN3,3,E9,8,D3,P,<NA>
252,TFMN3,3,F1,12,D3,P,<NA>
253,TFMN3,3,F2,12,E3,P,<NA>


In [11]:
other_df

,Experiment,Sequencing plate,Sequencing plate well,Robotic run plate,Robotic run plate well,Population or Single colony?,Sequencing sample
255,Gyorgy,3,F4,<NA>,<NA>,P,GB1
256,Gyorgy,3,F5,<NA>,<NA>,P,GB2
257,Gyorgy,3,F6,<NA>,<NA>,P,GB3
258,Gyorgy,3,F7,<NA>,<NA>,P,GB4
259,Gyorgy,3,F8,<NA>,<NA>,P,GB5
260,Gyorgy,3,F9,<NA>,<NA>,P,GB6
261,Gyorgy,3,F10,<NA>,<NA>,P,GB7


In [12]:
# Function to cycle wells list to length

def cycle_list_to_length(original_list, total_elements):
    from itertools import cycle, islice
    
    # Create a cycle iterator
    list_cycle = cycle(original_list)
    
    # Take only the required number of elements using islice
    new_list = list(islice(list_cycle, total_elements))
    
    return new_list

In [13]:
transfersamples = OD_df[['experiment', 'sample_name', 'transfer', 'well']]
transfersamples = transfersamples.loc[(~pd.isna(transfersamples['sample_name'])) & (transfersamples['transfer']!=0)]
transfersamples.drop_duplicates(inplace=True)

In [14]:
print(len(transfersamples.columns) == 4)
print(len(Nidhi_all_df.columns) == 7)

True
True


In [16]:
# Check that

## Selected robotic run plate transfer samples must be present in transfersamples
available = set(map(tuple, transfersamples[['experiment', 'transfer', 'well']].drop_duplicates().values))
required = set(map(tuple, Nidhi_df[['Experiment', 'Robotic run plate', 'Robotic run plate well']].drop_duplicates().values))
if len(required) - len(available)>0:
    print("The following selected robotic run plate transfer samples are missing from the OD_data:")
    for i in required-available:
        print(i)

## seqplates must be consecutive integers
min_seq_plate = min(Nidhi_all_df['Sequencing plate'].unique())
max_seq_plate = max(Nidhi_all_df['Sequencing plate'].unique())
if not all([x in Nidhi_all_df['Sequencing plate'].unique() for x in list(range(min_seq_plate, max_seq_plate+1))]):
    print("The sequencing plate indexes are not consecutive integers:")
    print(sorted(Nidhi_all_df['Sequencing plate'].unique()))

## Seqwells must be in wells
incorrect_wells = set(Nidhi_all_df['Sequencing plate well'].unique()) - set(LIMS_wells_byRow)
if len(incorrect_wells) > 0:
    print("The following selected sequencing plate well locations are incorrect:")
    print(incorrect_wells)

## Seqwells must be in consecutive order
expected_seq_plate_wells = cycle_list_to_length(LIMS_wells_byRow, len(Nidhi_all_df))
if not(Nidhi_all_df['Sequencing plate well'].to_list() == expected_seq_plate_wells):
    print("Sequencing plate wells are not in consecutive order.")
           
## All robotic run plate-well combos are unique
duplicated_roboplate_locs = Nidhi_df[['Experiment', 'Robotic run plate','Robotic run plate well']].duplicated()
if sum(duplicated_roboplate_locs)!=0:
    print("The following robotic run plate-well combos are non-unique:")
    display(Nidhi_df[['Experiment', 'Robotic run plate','Robotic run plate well']][duplicated_roboplate_locs])

## All seqplate-well combos are unique <-- This check would have caught TFMN1 mistake
duplicated_seqplate_locs = Nidhi_all_df[['Sequencing plate','Sequencing plate well']].duplicated()
if sum(duplicated_seqplate_locs)!=0:
    print("The following Sequencing plate-well combos are non-unique:")
    display(Nidhi_all_df[['Sequencing plate','Sequencing plate well']][duplicated_duplicated_seqplate_locs_locs])

In [23]:
seqsamples = pd.merge(transfersamples, Nidhi_df,
                      left_on=['experiment', 'transfer', 'well'],
                      right_on=['Experiment', 'Robotic run plate','Robotic run plate well'],
                      how='right')
seqsamples.drop(['experiment', 'transfer', 'well'], axis=1, inplace=True)

seqsamples.rename(columns={'sample_name': 'Sample Name'}, inplace=True)
seqsamples['Sequencing sample'] = seqsamples['Sample Name'] + '.' + 'T' + seqsamples['Robotic run plate'].astype(str) + '.' + seqsamples['Population or Single colony?']
seqsamples['Robotic run plate well row'] = seqsamples['Robotic run plate well'].apply(lambda x: x[0])
seqsamples['Robotic run plate well column'] = seqsamples['Robotic run plate well'].apply(lambda x: int(x[1:]))

seqsamples = pd.concat([seqsamples, other_df])
seqsamples['Sequencing plate well row'] = seqsamples['Sequencing plate well'].apply(lambda x: x[0])
seqsamples['Sequencing plate well column'] = seqsamples['Sequencing plate well'].apply(lambda x: int(x[1:]))

seqsamples = seqsamples[[
    'Sequencing sample',
    'Experiment',
    'Sample Name',
    'Sequencing plate',
    'Sequencing plate well',
    'Sequencing plate well row',
    'Sequencing plate well column',
    'Population or Single colony?',
    'Robotic run plate',
    'Robotic run plate well',
    'Robotic run plate well row',
    'Robotic run plate well column'
]]

byRow = ['Sequencing plate', 'Sequencing plate well row', 'Sequencing plate well column']
byCol = ['Sequencing plate', 'Sequencing plate well column', 'Sequencing plate well row']
seqsamples.sort_values(byRow, inplace=True)

In [24]:
seqsamples

,Sequencing sample,Experiment,Sample Name,Sequencing plate,Sequencing plate well,Sequencing plate well row,Sequencing plate well column,Population or Single colony?,Robotic run plate,Robotic run plate well,Robotic run plate well row,Robotic run plate well column
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb.2,1,A1,A,1,P,1,B2,B,2.0
4,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2,1,A2,A,2,P,1,B3,B,3.0
5,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2,1,A3,A,3,P,1,B4,B,4.0
6,TFMN2.ACN3500.Iva.gDNA3560.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3560.2,1,A4,A,4,P,1,B5,B,5.0
7,TFMN2.ACN3500.Iva.gDNA3575.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3575.2,1,A5,A,5,P,1,B6,B,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,GB3,Gyorgy,<NA>,3,F6,F,6,P,<NA>,<NA>,NaN,NaN
258,GB4,Gyorgy,<NA>,3,F7,F,7,P,<NA>,<NA>,NaN,NaN
259,GB5,Gyorgy,<NA>,3,F8,F,8,P,<NA>,<NA>,NaN,NaN
260,GB6,Gyorgy,<NA>,3,F9,F,9,P,<NA>,<NA>,NaN,NaN


In [25]:
path_to_seqsamples_table = '/storage/nspahr/RoboticRun_analysis/LIMS_seqsamples.csv'
seqsamples.to_csv(path_to_seqsamples_table, index=False)

## Download from seq facility, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and respective libraries.
- Copy all illumina fastqs into short lib, renaming in the process.

In [29]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'GFHFWC'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir, exist_ok=True)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
os.makedirs(home_dir, exist_ok=True)

/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-04_GFHFWC


In [31]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

ITEM GFHFWC
{'code': 'GFHFWC',
 'done_date': '2026-03-04T21:30:33.346779+00:00',
 'gross': 13886.0,
 'order_name': 'HANKE_260220',
 'product_name': 'custom_illumina',
 'quantity': 262,
 'status': 'complete'}



DOWNLOADING RESULTS FOR GFHFWC 

No results found for GFHFWC: b'{"message": "Error getting item results link"}\n'
DOWNLOADING READS FOR GFHFWC 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-04_GFHFWC/GFHFWC_reads.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-04_GFHFWC/GFHFWC_reads.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-04_GFHFWC/GFHFWC_reads


In [ ]:
### TODO: How to ensure that archive was successfully unzipped?? Pipeline log file into home_dir?

In [36]:
# Ensure that you unzipped the expected number of fastq files

plasmidsaurus_read_folder_name = os.path.join(reception_dir, item_code + '_reads')
exp_n_samples = 262
n_fastqfiles = len([x for x in os.listdir(plasmidsaurus_read_folder_name) if x.endswith(".fastq.gz")])
print(exp_n_samples*2 == n_fastqfiles)

True


In [92]:
# Spot-check if duplicate read ID problem is fixed

plasmidsaurus_read_folder_name = item_code + '_reads'
os.listdir(os.path.join(reception_dir, plasmidsaurus_read_folder_name))

['GFHFWC_261_GB6_R1.fastq.gz',
 'GFHFWC_261_GB6_R2.fastq.gz',
 'GFHFWC_262_GB7_R1.fastq.gz',
 'GFHFWC_262_GB7_R2.fastq.gz',
 'GFHFWC_46_TFMN2.ACN3500.Iva.DEL6kb.3.T2.P_R1.fastq.gz',
 'GFHFWC_46_TFMN2.ACN3500.Iva.DEL6kb.3.T2.P_R2.fastq.gz',
 'GFHFWC_47_TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3.T2.P_R1.fastq.gz',
 'GFHFWC_47_TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3.T2.P_R2.fastq.gz',
 'GFHFWC_48_TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T2.P_R1.fastq.gz',
 'GFHFWC_48_TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T2.P_R2.fastq.gz',
 'GFHFWC_49_TFMN2.ACN3500.Iva.gDNA3560.3.T2.P_R1.fastq.gz',
 'GFHFWC_49_TFMN2.ACN3500.Iva.gDNA3560.3.T2.P_R2.fastq.gz',
 'GFHFWC_50_TFMN2.ACN3500.Iva.gDNA3575.3.T2.P_R1.fastq.gz',
 'GFHFWC_50_TFMN2.ACN3500.Iva.gDNA3575.3.T2.P_R2.fastq.gz',
 'GFHFWC_51_TFMN2.ACN3500.Iva.gDNA3749.3.T2.P_R1.fastq.gz',
 'GFHFWC_51_TFMN2.ACN3500.Iva.gDNA3749.3.T2.P_R2.fastq.gz',
 'GFHFWC_52_TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.3.T2.P_R1.fastq.gz',
 'GFHFWC_52_TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.3.T

In [38]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'GFHFWC_79_TFMN2.ACN3500.Mxb.gDNA3749.2.T2.P_R1.fastq.gz')} | head

@LH01025:78:23FHGFLT3:8:1101:5733:1064 1:N:0:TAGCATAACC+TGCGCATAGC
ATTCNAGAGTTCAATTTTGAAATCAGGATTATCGTATTTGGGTGAAAGTGTTTTAGTTTCTAACGCTCGGATAATTTTAGAAAGGCTAGTTTTTCCAGAGTAGTTTCGACCATAAAAAATATTGATATCTTTAAAGTCATAAGTTTGGTTC
+
IIII#IIIII9III9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@LH01025:78:23FHGFLT3:8:1101:8720:1080 1:N:0:TAGCATAACC+TGCGCATAGC
GGTGNTACAGCCTAAACTGATTATTCTGGATGAACCGACATCTGCACTTGATCGTACAACCCAGCGTGCAATTGTAAAACTACTGCGACATTTACAACAGCAAGAAAATCTAAGCTATTTATTCATCAGTCATGATTTGCAGGTCATACGT
+
IIII#99I9IIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIII9IIIIIIIII9IIIIIIIII9III9II9IIIII9II9III9II9III9II9IIIIIIIIIIIIII*IIIIIIIIIII9
@LH01025:78:23FHGFLT3:8:1101:9441:1080 1:N:0:TAGCATAACC+TGCGCATAGC
TCCANATACCACCTGGCATTTTGAAGGTTGACTGCGCATGCAATTCTGGGCGTAATTTACGATAACGCAGATAGCAAACCATAATCATGCCCCACACACTAATAAACAGGATCACGCAAAGCGAACTAGCCAGCGTAAATGCCTCCATGGT

gzip: stdout: Broken pipe


In [39]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'GFHFWC_79_TFMN2.ACN3500.Mxb.gDNA3749.2.T2.P_R1.fastq.gz')} | grep "@LH01025:78:23FHGFLT3:8:1101:5733:1064 1:N:0:TAGCATAACC+TGCGCATAGC"


@LH01025:78:23FHGFLT3:8:1101:5733:1064 1:N:0:TAGCATAACC+TGCGCATAGC


**Comment:**

- In this spot check, only found the read ID once in this file. I assume this means that we are not dealing with the same read duplication problem.

In [93]:
from aisynbiopipeline.workflows.fastq_utils import create_manifest, parse_illumina_fastq_filename

folder = os.path.join(reception_dir, plasmidsaurus_read_folder_name)

manifest = create_manifest(folder, platform='plasmidsaurus_illumina')
manifest.head()

,sample_name,fwd_fastq,rvs_fastq
0,GFHFWC_100_TFMN2.ACN3500.Iva.gDNA3575-gDNA3749...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,GFHFWC_101_TFMN2.ACN3500.Van.DEL6kb.1.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,GFHFWC_102_TFMN2.ACN3500.Van.DEL6kb.2.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,GFHFWC_103_TFMN2.ACN3500.Iva.DEL6kb.3.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,GFHFWC_104_TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3....,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [41]:
len(manifest)

262

In [42]:
# Plasmidsaurus provides a sort of sample manifest for download from the seqorder page. (not available with read download through API).
# Downloaded to my laptop, uploaded to home_dir, now copying to reception dir.

import shutil

shutil.copy2(os.path.join(home_dir, f'{item_code}-summary-report.csv'), reception_dir)

'/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-04_GFHFWC/GFHFWC-summary-report.csv'

In [43]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
# long = Library(seqorder, 'Nanopore', create=True)

In [45]:
# Copy fastqs into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['fwd_fastq'])
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['rvs_fastq'])
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

## Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [414]:
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

seqorder = SeqOrder('Plasmidsaurus_2026-03-04_GFHFWC')
short = Library(seqorder, 'Illumina')

In [26]:
experiments = ['TFMN2', 'TFMN3', 'Gyorgy']

In [6]:
short_manifest = short.create_manifest('received')

In [7]:
short_manifest

,sample_name,R1,R2
0,GB1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,GB2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,GB3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,GB4,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,GB5,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
...,...,...,...
257,TFMN3.ACN3749.Mxb.noDNA.4.T8.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
258,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
259,TFMN3.ACN3749.Mxb.noDNA.5.T12.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
260,TFMN3.ACN3749.Mxb.noDNA.5.T5.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [8]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [30]:
# Cross-checking seq sample names

thisExpLIMSseqsamples_short = pd.DataFrame()
for e in experiments:
    thisExpLIMSseqsamples_short = pd.concat(
        [thisExpLIMSseqsamples_short,
         query_lims('Seqsamples', filters={'Experiment': e})],
        axis=0
    )
LIMSseqsample_names = thisExpLIMSseqsamples_short['Sequencing_sample'].to_list()

print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in LIMSseqsample_names for x in short_manifest['sample_name'].to_list()]))

Are all short Plasmidsaurus seqsamples from order GFHFWC in the LIMS?
True


In [40]:
# Moving Gyorgy seqsamples into ~/storage/tmp/Gyorgy for zip and download to Box

import shutil

g_dir = '/storage/nspahr/tmp/Gyorgy'
os.makedirs(g_dir, exist_ok=True)

gyorgy_LIMS_seqsamples = query_lims(
    'Seqsamples',
    filters={'Experiment': 'Gyorgy'}
)['Sequencing_sample'].to_list()

gyorgy_seqsamples = [SeqSample(short, x) for x in gyorgy_LIMS_seqsamples]

for s in gyorgy_seqsamples:
    print(s.sample_name)
    print(s.received)

GB1
[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/received/GB1_R1.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/received/GB1_R2.fastq.gz')]
GB2
[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/received/GB2_R1.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/received/GB2_R2.fastq.gz')]
GB3
[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/received/GB3_R1.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsauru

In [42]:
for s in gyorgy_seqsamples:
    for i in s.received:
        shutil.copy2(i, g_dir)

In [61]:
archive_path = shutil.make_archive(os.path.join(g_dir, 'Gyorgy'), 'zip', root_dir=g_dir)
# print(f"Created archive: {archive_path}")

In [711]:
# Create a measurements table (formatted for LIMS) for upload into LIMS

experiments = ['TFMN2', 'TFMN3']

LIMSseqsamples = pd.DataFrame()
for e in experiments:
    LIMSseqsamples = pd.concat(
        [LIMSseqsamples,
         query_lims('Seqsamples', filters={'Experiment': e})],
        axis=0
    )
LIMSseqsamples = LIMSseqsamples[['Sequencing_sample', 'Experiment', 'Sample_Name']]

LIMSseqsamples

,Sequencing_sample,Experiment,Sample_Name
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb.2
1,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2
2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2
3,TFMN2.ACN3500.Iva.gDNA3560.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3560.2
4,TFMN2.ACN3500.Iva.gDNA3575.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3575.2
...,...,...,...
13,TFMN3.ACN3749.Mxb.noDNA.5.T8.P,TFMN3,TFMN3.ACN3749.Mxb.noDNA.5
14,TFMN3.ACN3749.Mxb.noDNA.2.T12.P,TFMN3,TFMN3.ACN3749.Mxb.noDNA.2
15,TFMN3.ACN3749.Mxb.noDNA.3.T12.P,TFMN3,TFMN3.ACN3749.Mxb.noDNA.3
16,TFMN3.ACN3749.Mxb.noDNA.4.T12.P,TFMN3,TFMN3.ACN3749.Mxb.noDNA.4


In [78]:
measurements = LIMSseqsamples.copy()
measurements['Type'] = 'Short_DNA_reads'
measurements['Protocol'] = pd.NA
measurements['Who measured'] = 'Paul Hanke & technicians'
measurements['Lab'] = 'ANL & Plasmidsaurus'
measurements['Timestamp'] = '03-04-2026'

measurements.rename(columns={
    'Sequencing_sample': 'Name',
    'Sample_Name': 'Sample ID'
}, inplace=True)

measurements = measurements[['Name', 'Type', 'Experiment', 'Sample ID', 'Protocol', 'Who measured', 'Lab', 'Timestamp']]
measurements.to_csv(f'/storage/nspahr/tmp/{item_code}_measurements.csv')

In [80]:
# Creating batch (list) of seqsamples for this seqorder

seqsamples = [SeqSample(short, row['Sequencing_sample']) for _, row in LIMSseqsamples.iterrows()]

## QA/QC

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.fastp_task 1
"""

In [ ]:
## fastp workers running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 2

In [82]:
# Create the trimmed subfolder
short.create_subfolder('trimmed')

In [83]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [111]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in seqsamples + gyorgy_seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [115]:
for i in results:
    print(i.status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS


In [114]:
all([(r.status=='SUCCESS') for r in results])

True

In [116]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, item_code + '_trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)


/// ]8;id=897930;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.33 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-04_GFHFWC/Plasmidsaurus_2026-03-04_GFHFWC_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 1049/1049                                                   html

             fastp | Found 262 reports
     write_results | Data        : multiqc_data   (overwritten)
     write_results | Report      : multiqc_report.html   (overwritten)
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-03-04_GFHFWC/GFHFWC_trimmed_multiqc_report.html'

Then check QC report, download, and add to LIMS in data > short_reads > QAQC_reports, and link to from Seqorders table.

In [140]:
readN = pd.read_csv("/storage/nspahr/tmp/last_seqorder_read_numbers.csv", sep='csv')

/tmp/ipykernel_447898/2057689832.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  readN = pd.read_csv("/storage/nspahr/tmp/last_seqorder_read_numbers.csv", sep='csv')


In [141]:
readN

,"sample,pct_duplication,n_reads,gc,pct_filtered,pct_adapter"
0,"GB1,2.00%,4.5M,41.60%,98.80%,8.50%"
1,"GB2,2.40%,7.4M,47.10%,98.70%,7.80%"
2,"GB3,1.50%,4.7M,58.90%,98.60%,5.80%"
3,"GB4,2.60%,15.5M,60.20%,98.40%,8.90%"
4,"GB5,1.20%,5.7M,58.50%,98.50%,9.80%"
...,...
257,"TFMN3.ACN3749.Mxb.noDNA.4.T8.P,3.70%,13.8M,40...."
258,"TFMN3.ACN3749.Mxb.noDNA.5.T1.P,6.20%,27.4M,40...."
259,"TFMN3.ACN3749.Mxb.noDNA.5.T12.P,4.90%,21.7M,40..."
260,"TFMN3.ACN3749.Mxb.noDNA.5.T5.P,4.80%,19.8M,40...."


In [166]:
def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN2\.(ACN3500|ACN3575)\.(Iva|Mxb|Van)\.(DEL6kb|DEL6kb-DELvanK|DEL6kb-DELvanK-DELadeK|gDNA3560|gDNA3575|gDNA3749|gDNA3575-gDNA3749|gDNA3575-gDNA3749-gDNA3560)\.([1-8]))\.T(\d{1,2})\.P$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        strain = str(match.group(2))
        media = str(match.group(3))
        construct = str(match.group(4))
        replicate = str(match.group(5))
        transfer = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        strain = 'NA'
        media = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'

    return {'sample': sample, 'strain': strain, 'media': media, 'construct': construct, 'replicate': replicate, 'transfer': transfer}
        

In [152]:
samples = input().split(' ')

 TFMN2.ACN3500.Iva.DEL6kb.2.T1.P TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.4.T1.P TFMN2.ACN3500.Mxb.gDNA3575-gDNA3749-gDNA3560.1.T2.P TFMN2.ACN3500.Iva.gDNA3560.3.T1.P TFMN2.ACN3500.Mxb.DEL6kb-DELvanK.3.T2.P TFMN2.ACN3500.Van.gDNA3575-gDNA3749.1.T2.P TFMN2.ACN3500.Iva.DEL6kb-DELvanK.4.T1.P TFMN2.ACN3500.Iva.DEL6kb.2.T3.P TFMN2.ACN3500.Iva.gDNA3749.3.T3.P TFMN2.ACN3500.Iva.gDNA3560.4.T1.P TFMN2.ACN3500.Iva.DEL6kb.4.T1.P TFMN2.ACN3500.Iva.DEL6kb-DELvanK.4.T3.P TFMN2.ACN3500.Iva.gDNA3575.3.T3.P TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P TFMN2.ACN3500.Van.DEL6kb-DELvanK.1.T3.P TFMN2.ACN3500.Iva.DEL6kb.2.T2.P TFMN2.ACN3500.Van.DEL6kb.1.T7.P TFMN2.ACN3500.Van.gDNA3749.1.T3.P TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T7.P TFMN2.ACN3500.Iva.DEL6kb.3.T3.P TFMN2.ACN3500.Van.gDNA3575-gDNA3749.1.T7.P TFMN2.ACN3500.Van.DEL6kb-DELvanK.2.T7.P TFMN2.ACN3500.Iva.gDNA3575.4.T13.P TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3.T1.P TFMN2.ACN3500.Iva.DEL6kb.3.T1.P TFMN2.ACN3500.Iva.gDNA3749.3.T7.P TFMN2.ACN3500.Van.DEL6kb-DE

In [153]:
samples

['TFMN2.ACN3500.Iva.DEL6kb.2.T1.P',
 'TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.4.T1.P',
 'TFMN2.ACN3500.Mxb.gDNA3575-gDNA3749-gDNA3560.1.T2.P',
 'TFMN2.ACN3500.Iva.gDNA3560.3.T1.P',
 'TFMN2.ACN3500.Mxb.DEL6kb-DELvanK.3.T2.P',
 'TFMN2.ACN3500.Van.gDNA3575-gDNA3749.1.T2.P',
 'TFMN2.ACN3500.Iva.DEL6kb-DELvanK.4.T1.P',
 'TFMN2.ACN3500.Iva.DEL6kb.2.T3.P',
 'TFMN2.ACN3500.Iva.gDNA3749.3.T3.P',
 'TFMN2.ACN3500.Iva.gDNA3560.4.T1.P',
 'TFMN2.ACN3500.Iva.DEL6kb.4.T1.P',
 'TFMN2.ACN3500.Iva.DEL6kb-DELvanK.4.T3.P',
 'TFMN2.ACN3500.Iva.gDNA3575.3.T3.P',
 'TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P',
 'TFMN2.ACN3500.Van.DEL6kb-DELvanK.1.T3.P',
 'TFMN2.ACN3500.Iva.DEL6kb.2.T2.P',
 'TFMN2.ACN3500.Van.DEL6kb.1.T7.P',
 'TFMN2.ACN3500.Van.gDNA3749.1.T3.P',
 'TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T7.P',
 'TFMN2.ACN3500.Iva.DEL6kb.3.T3.P',
 'TFMN2.ACN3500.Van.gDNA3575-gDNA3749.1.T7.P',
 'TFMN2.ACN3500.Van.DEL6kb-DELvanK.2.T7.P',
 'TFMN2.ACN3500.Iva.gDNA3575.4.T13.P',
 'TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3.T1.P'

In [167]:
rows = []

for i in samples:
    row = {}
    row['seqsample'] = i
    row.update(parse_seqsample_name(i))
    rows.append(row)

df = pd.DataFrame(rows)
df

,seqsample,sample,strain,media,construct,replicate,transfer
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,TFMN2.ACN3500.Iva.DEL6kb.2,ACN3500,Iva,DEL6kb,2,1
1,TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.4.T1.P,TFMN2.ACN3500.Iva.gDNA3575-gDNA3749.4,ACN3500,Iva,gDNA3575-gDNA3749,4,1
2,TFMN2.ACN3500.Mxb.gDNA3575-gDNA3749-gDNA3560.1...,TFMN2.ACN3500.Mxb.gDNA3575-gDNA3749-gDNA3560.1,ACN3500,Mxb,gDNA3575-gDNA3749-gDNA3560,1,2
3,TFMN2.ACN3500.Iva.gDNA3560.3.T1.P,TFMN2.ACN3500.Iva.gDNA3560.3,ACN3500,Iva,gDNA3560,3,1
4,TFMN2.ACN3500.Mxb.DEL6kb-DELvanK.3.T2.P,TFMN2.ACN3500.Mxb.DEL6kb-DELvanK.3,ACN3500,Mxb,DEL6kb-DELvanK,3,2
...,...,...,...,...,...,...,...
257,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2.T20.P,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2,ACN3500,Iva,DEL6kb-DELvanK-DELadeK,2,20
258,TFMN3.ACN3749.Mxb.noDNA.3.T1.P,NA,NA,NA,NA,NA,NA
259,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,NA,NA,NA,NA,NA,NA
260,TFMN3.ACN3749.Mxb.noDNA.3.T8.P,NA,NA,NA,NA,NA,NA


In [174]:
df['reads'] = input().split(' ')

 0 0 0 0.00001 0.00001 0.00001 0.00067 0.01445 0.01596 0.01843 0.02129 0.0253 0.09985 0.13184 0.14022 0.15347 0.22701 0.27155 0.29075 0.42793 0.64868 0.65603 0.66171 0.72094 0.74701 0.81297 0.82872 0.92818 1.00478 1.06137 1.16368 1.44675 1.46533 1.48468 1.49162 1.50867 1.56275 1.57456 1.57961 1.61368 1.63918 1.65127 1.65148 1.65809 1.66067 1.66345 1.67146 1.73119 1.80131 1.82807 1.88119 1.94432 1.99453 2.00401 2.11306 2.13012 2.23069 2.23535 2.23776 2.26331 2.32406 2.33846 2.35622 2.38916 2.40506 2.46621 2.50077 2.50798 2.51853 2.57059 2.59991 2.66697 2.69976 2.72449 2.73834 2.74227 2.76898 2.82077 2.82122 2.87993 2.89873 2.91433 2.96663 3.202 3.22083 3.30978 3.31497 3.33184 3.36962 3.43162 3.44899 3.45556 3.47166 3.51997 3.58318 3.66515 3.6751 3.74493 3.7524 3.78531 3.83102 3.92714 3.98436 4.10223 4.11639 4.12097 4.15001 4.16795 4.27582 4.31975 4.41145 4.43724 4.44274 4.46345 4.5246 4.54331 4.60146 4.67516 4.72467 4.75817 4.95347 4.98223 5.12678 5.1522 5.15571 5.16698 5.17744 5.22743 

In [287]:
experiments =['TFMN2', 'TFMN3', 'Gyorgy']
plate_loc = pd.DataFrame()
for e in experiments:
    plate_loc = pd.concat(
        [plate_loc,
         query_lims('Seqsamples', filters={'Experiment': e})],
        axis=0
    )

In [288]:
plate_loc = plate_loc[['Sequencing_sample', 'Sequencing_plate', 'Sequencing_plate_well_row', 'Sequencing_plate_well_column', 'Robotic_run_plate', 'Robotic_run_plate_well', 'Robotic_run_plate_well_row', 'Robotic_run_plate_well_column']]

In [289]:
plate_loc.head()

,Sequencing_sample,Sequencing_plate,Sequencing_plate_well_row,Sequencing_plate_well_column,Robotic_run_plate,Robotic_run_plate_well,Robotic_run_plate_well_row,Robotic_run_plate_well_column
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,1,A,1,1,B2,B,2
1,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,1,A,2,1,B3,B,3
2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2.T1.P,1,A,3,1,B4,B,4
3,TFMN2.ACN3500.Iva.gDNA3560.2.T1.P,1,A,4,1,B5,B,5
4,TFMN2.ACN3500.Iva.gDNA3575.2.T1.P,1,A,5,1,B6,B,6


In [177]:
df = df.sort_values('seqsample')
df

,seqsample,sample,strain,media,construct,replicate,transfer,reads
115,GB1,NA,NA,NA,NA,NA,NA,4.54331
179,GB2,NA,NA,NA,NA,NA,NA,7.39233
117,GB3,NA,NA,NA,NA,NA,NA,4.67516
230,GB4,NA,NA,NA,NA,NA,NA,15.53441
144,GB5,NA,NA,NA,NA,NA,NA,5.74532
...,...,...,...,...,...,...,...,...
224,TFMN3.ACN3749.Mxb.noDNA.4.T8.P,NA,NA,NA,NA,NA,NA,13.82185
259,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,NA,NA,NA,NA,NA,NA,27.44412
245,TFMN3.ACN3749.Mxb.noDNA.5.T12.P,NA,NA,NA,NA,NA,NA,21.66614
238,TFMN3.ACN3749.Mxb.noDNA.5.T5.P,NA,NA,NA,NA,NA,NA,19.84106


In [194]:
df.drop('plate', axis=1, inplace=True)

In [188]:
df['reads'] = df['reads'].astype(float)

In [290]:
merge = pd.merge(df, plate_loc, left_on='seqsample', right_on='Sequencing_sample')
merge

,seqsample,sample,strain,media,construct,replicate,transfer,reads,Sequencing_sample,Sequencing_plate,Sequencing_plate_well_row,Sequencing_plate_well_column,Robotic_run_plate,Robotic_run_plate_well,Robotic_run_plate_well_row,Robotic_run_plate_well_column
0,GB1,NA,NA,NA,NA,NA,NA,4.54331,GB1,3,F,4,,,,
1,GB2,NA,NA,NA,NA,NA,NA,7.39233,GB2,3,F,5,,,,
2,GB3,NA,NA,NA,NA,NA,NA,4.67516,GB3,3,F,6,,,,
3,GB4,NA,NA,NA,NA,NA,NA,15.53441,GB4,3,F,7,,,,
4,GB5,NA,NA,NA,NA,NA,NA,5.74532,GB5,3,F,8,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,TFMN3.ACN3749.Mxb.noDNA.4.T8.P,NA,NA,NA,NA,NA,NA,13.82185,TFMN3.ACN3749.Mxb.noDNA.4.T8.P,3,E,10,8,E3,E,3
258,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,NA,NA,NA,NA,NA,NA,27.44412,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,3,E,2,1,F3,F,3
259,TFMN3.ACN3749.Mxb.noDNA.5.T12.P,NA,NA,NA,NA,NA,NA,21.66614,TFMN3.ACN3749.Mxb.noDNA.5.T12.P,3,F,3,12,F3,F,3
260,TFMN3.ACN3749.Mxb.noDNA.5.T5.P,NA,NA,NA,NA,NA,NA,19.84106,TFMN3.ACN3749.Mxb.noDNA.5.T5.P,3,E,7,5,F3,F,3


In [291]:
merge.sort_values(['Robotic_run_plate', 'Robotic_run_plate_well_row', 'Robotic_run_plate_well_column']).to_csv("/storage/nspahr/tmp/reads_location_corr.csv")

In [292]:
merge.groupby(['Sequencing_plate'])['reads'].mean()

Sequencing_plate
1     4.156933
2     4.442481
3    14.280861
Name: reads, dtype: float64

## Run Breseq

- no subsampling
- against ACN3500

In [458]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk',
 'ACN3500_NSS.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
cd ~/code/AISynbioPipeline
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# seven breseq workers are running!

In [207]:
# Create breseq dir

short.create_subfolder('breseq')

In [208]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,GB1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,GB2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,GB3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,GB4,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,GB5,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
...,...,...,...
257,TFMN3.ACN3749.Mxb.noDNA.4.T8.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
258,TFMN3.ACN3749.Mxb.noDNA.5.T1.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
259,TFMN3.ACN3749.Mxb.noDNA.5.T12.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
260,TFMN3.ACN3749.Mxb.noDNA.5.T5.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [209]:
# Where should this code go?

# Specifies and assigns breseq parameters

from pathlib import Path

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4, polymorphism_frequency_cutoff=0.05):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors,
        'polymorphism_frequency_cutoff': polymorphism_frequency_cutoff
        
    }
    return breseq_params

In [210]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

results = []

for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk', polymorphism_frequency_cutoff=0.005, fold_coverage=0),
        queue='breseq'
    )
    results.append(result)    

In [215]:
sum([(r.status=='SUCCESS') for r in results])

0

In [214]:
for i in results:
    print(i.status)
for i in results:
    print(i.result['output']) ## Check version_name for breseq run

PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING


TypeError: 'NoneType' object is not subscriptable

In [229]:
breseq_dir = os.path.join(home_dir, 'breseq_0b03256c7a')
os.makedirs(breseq_dir, exist_ok=True)

In [230]:
# Symlink to library breseq folder
output_symlinks_dir = os.path.join(breseq_dir, 'symlink_to_library_breseq_folder')
os.makedirs(output_symlinks_dir)

path_to_folder = short.path / 'breseq'
dst = os.path.join(output_symlinks_dir, 'breseq')
os.symlink(path_to_folder, dst)

In [248]:
# Forgot to run the parent samples for these experiments

from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

parentlib = Library(SeqOrder('Plasmidsaurus_2026-01-13_S2SVV5'), "Illumina")
parent_3500 = SeqSample(parentlib, 'ANL.stock.ACN3500.colony2')

parentlib = Library(SeqOrder('Plasmidsaurus_9-13-2025_HTGS8F'), "Illumina")
parent_3575 = SeqSample(parentlib, 'ANLstock.ACN3575.colony1')

parentlib = Library(SeqOrder('Plasmidsaurus_2026-02-13_QYYTMB'), "Illumina")
parent_3749 = SeqSample(parentlib, 'ANLstock.ACN3749.colony1')

parents = [parent_3500, parent_3575, parent_3749]

for p in parents:
    print(os.listdir(p.breseq))

['breseq_06bcbb18f5', '.breseq_params_registry.json', 'breseq_0d5e3dff7b', 'breseq_df1972644b']
['breseq_df1972644b', '.breseq_params_registry.json']
['breseq_df1972644b', '.breseq_params_registry.json']


In [249]:
# Submit tasks

results = []

for sample in parents:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk', polymorphism_frequency_cutoff=0.005, fold_coverage=0),
        queue='breseq'
    )
    results.append(result)

In [298]:
sum([(r.status=='SUCCESS') for r in results])

2

In [300]:
for i in results:
    print(i.status)

FAILURE
SUCCESS
SUCCESS


In [251]:
breseq_version_name = 'breseq_0b03256c7a'

In [301]:
for s in parents:
    print(Breseq.from_existing(s.breseq/breseq_version_name).exists)

False
True
True


In [302]:
# Rerunning parent_3500 (hit timeout limit)

results = []

for sample in [parent_3500]:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk', polymorphism_frequency_cutoff=0.005, fold_coverage=0),
        queue='breseq'
    )
    results.append(result)

In [321]:
for i in results:
    print(i.status)

SUCCESS


In [322]:
os.listdir(parent_3500.breseq/breseq_version_name/'output')

['log.txt',
 'calibration',
 'evidence',
 'output.gd',
 'output.vcf',
 'summary.json',
 'index.html',
 'marginal.html',
 'summary.html',
 'output.done']

## Breseq analysis TFMN2

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.

In [774]:
exp = 'TFMN2'

In [775]:
# Many of the samples had a low number of reads. Making a subbatch of seqsamples with at least 300X coverage for downstream analysis.

from aisynbiopipeline.workflows.breseq import Breseq

seqsamples = [SeqSample(short, row['Sequencing_sample']) for _, row in LIMSseqsamples.iterrows()]
low_cov_samples = [s for s in seqsamples if Breseq.from_existing(s.breseq/breseq_version_name).avg_coverage<100]
suff_cov_samples = [s for s in seqsamples if s not in low_cov_samples]
print(f"Number of seqsamples with at least 200x coverage: {len(suff_cov_samples)}/{len(seqsamples)}")

Number of seqsamples with at least 200x coverage: 189/255


In [776]:
seqsamples = [s for s in suff_cov_samples if exp in s.sample_name]
print(len(seqsamples))
seqsamples.append(parent_3500)
print(len(seqsamples))

171
172


In [272]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [777]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN2\.(ACN3500|ACN3575|ACN3749)\.(Iva|Mxb|Van)\.(DEL6kb|DEL6kb-DELvanK|DEL6kb-DELvanK-DELadeK|gDNA3560|gDNA3575|gDNA3749|gDNA3575-gDNA3749)\.([1-8]))\.T(\d{1,2})\.P$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        strain = str(match.group(2))
        media = str(match.group(3))
        construct = str(match.group(4))
        replicate = str(match.group(5))
        transfer = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        strain = 'NA'
        media = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'

    return {'sample': sample, 'strain': strain, 'media': media, 'construct': construct, 'replicate': replicate, 'transfer': transfer}
        

In [778]:
from collections import Counter
from aisynbiopipeline.workflows.breseq import Breseq
from aisynbiopipeline.workflows.mapping import PileupCol
import numpy as np

def create_breseq_summary(seqsample_batch, version_name, output_path=None, regions=None, loci=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        
        if loci:
            for key, value in loci['mutations'].items():
                prefix = key
                basecalls = PileupCol(b.bam_path, value['locus']).basecalls
                locus_cov = len(basecalls)
                counts = Counter(basecalls)
                try:
                    alt_freq = counts[value['alt_allele']] / locus_cov
                except ZeroDivisionError:
                    alt_freq = None
                row[key+'_locus_cov'] = locus_cov
                row[key+'_alleleCounts'] = dict(counts)
                row[key+'_alt_allele_freq'] = alt_freq
        
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    breseq_summary['is_parent'] = breseq_summary['seqsample'].apply(lambda x: x.startswith('ANL'))
    
    # Put rows and cols into correct order
    metadata_cols = ['seqsample', 'sample', 'strain', 'media', 'construct', 'replicate', 'transfer', 'is_parent']
    other_cols = [col for col in breseq_summary.columns.to_list() if col not in metadata_cols]
    breseq_summary = breseq_summary[metadata_cols + other_cols]
    breseq_summary.sort_values(
        ['is_parent', 'media', 'construct', 'strain', 'replicate', 'transfer'],
        ascending=[False, True, True, True, True, True],
        inplace=True
    )

    # Write breseq run summary to csv
    if output_path:
        breseq_summary.to_csv(output_path, index=False)
    
    return breseq_summary

In [784]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB'),
    'verAB': get_region_parameter(genome3500, 'verA', 'verB')
}

In [838]:
regions

{'ver_cassette': 'ACN3500_NSS:941311-949904',
 'verAB': 'ACN3500_NSS:947838-949904'}

In [341]:
# loci = {
#     'reference': 'ACN3500_NSS',
#     'mutations': {
#         'promoter_6kb.DEL': {
#             'locus': (950040, 955945),  # added 50 bp from start and subtracted 50 bp from end to accomodate slightly different start and end loci between samples
#             'ref_allele': '...',
#             'alt_allele': ''
#         },
#         'vanK_1-bp.DEL': {
#             'locus': (974618, 974618),
#             'ref_allele': 'G',
#             'alt_allele': ''
#         },
#         'ACIAD_RS08195<->fecI.SNP': {
#             'locus': (1405240, 1405241),  ### THIS IS WRONG!!!
#             'ref_allele': 'T',
#             'alt_allele': 'A'
#         },
#         'dsbD.Q75stop': {
#             'locus': (3465740, 3465740),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'iscR_2-bp.DEL': {
#             'locus': (1405240, 1405241),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'rpoD_3-bp.DEL': {
#             'locus': (2860637, 2860639),
#             'ref_allele': 'GAA',
#             'alt_allele': ''
#         },
#         'adeK_2-bp.DEL': {
#             'locus': (2883783, 2883784),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'mnmA.R245H': {
#             'locus': (1230459, 1230459),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'ACIAD_RS01630_2-bp.DEL': {
#             'locus': (345201, 345202),
#             'ref_allele': 'AA',
#             'alt_allele': ''
#         }
#     }
# }

In [791]:
regions_to_include = ['ver_cassette', 'verAB']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

breseq_folder = home_dir + '/' + exp + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

breseq_summary = create_breseq_summary(
    seqsamples,
    breseq_version_name,
    output_path=os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_summary.csv'),
    regions=regions_sub,
    # loci=loci
)

# create_html_comparison(
#     seqsample_batch,
#     breseq_version_name,
#     os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.html')
# )

In [793]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

breseq_objects = []
for s in seqsamples:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ breseq_version_name
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

reference = 'ACN3500_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

breseq_outfolder = home_dir + '/' + exp + '/' + breseq_version_name

table_format = 'html'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

In [794]:
compare_df = pd.read_csv(csv)

In [795]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Need to exclude these because they will mess with pivoting on title
cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies',
    ]

# Exclude these because they are not of interest
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment', 'transl_table']

cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

# Columns that should only have integer values. Some may have weird text which messes with conversion to nullable int type.
# Force to numeric
int_cols = [
    'aa_position',
    'codon_number',
    'codon_position',
    'gene_position',
    'insert_position',
    'position',
    'position_end',
    'position_start',
    'repeat_length',
    'size',
]
for col in int_cols:
    compare_df[col] = pd.to_numeric(compare_df[col], errors='coerce')

# Now that expected numeric cols are numeric, can convert to int if possible
compare_df = compare_df[cols_to_keep].convert_dtypes()

# Define columns that will form the Multiindex 
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

# Create the pivoted mutation frame, where distinct mutations are rows, samples are columns, values are frequencies
df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)

# Order the comparison just like the summary
# df = df.loc[~(df > 0).all(axis=1)] # Only mutations that don't appear in all samples
df = df[breseq_summary['seqsample'].to_list()]

In [453]:
# df_greater5 = df.loc[(df > 0.05).any(axis=1)].dropna(how="all")
# df_greater80 = df.loc[(df > 0.80).any(axis=1)].dropna(how="all")

In [377]:
# # Write reformatted mutations to csv.
# df.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted.csv'))
# df_greater5.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater5.csv'))
# df_greater80.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater80.csv'))

In [836]:
# ------ Make highlight color dictionary ------
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

def create_highlight_color_dict(index_col_names, breseq_summary):
    
    gray_fill = PatternFill(start_color="DDDDDD",
                            end_color="DDDDDD",
                            fill_type="solid")
    
    green_fill = PatternFill(start_color="E5FFCC",
                            end_color="E5FFCC",
                            fill_type="solid")
    
    white_fill = PatternFill(start_color="FFFFFF",
                            end_color="FFFFFF",
                            fill_type="solid")
    
    orange_fill = PatternFill(start_color="FFE5CC",
                            end_color="FFE5CC",
                            fill_type="solid")
    
    # ------ Assign column colors ------
    ## ALE transfersamples will be grouped by original sample and highlighted in alternating colors
    alt_colors = dict(
        zip(
            breseq_summary['sample'].unique(),
            len(breseq_summary['sample'].unique())*[white_fill, orange_fill]
        )
    )

    parents = breseq_summary.loc[breseq_summary['is_parent']]['seqsample'].to_list()
    # parent_sample_names = [x.replace("ANL.stock", "ANLstock") for x in parents]
    
    ALEsamples = breseq_summary.loc[~breseq_summary['is_parent']]
    ALEsample_colors = dict(zip(ALEsamples['seqsample'].to_list(), ALEsamples['sample'].apply(lambda x: alt_colors[x]).to_list()))
    
    highlight_colors = dict()
    highlight_colors.update(dict(zip(index_col_names, [gray_fill]*len(index))))
    highlight_colors.update(dict(zip(parents, [green_fill]*len(parents))))
    highlight_colors.update(ALEsample_colors)
    
    return highlight_colors


# ------ Build workbook (all samples, individual samples) ------

def write_mutation_comparison_wb(df, breseq_summary, excel_path):

    def get_parent_sample(sample):
        LIMSsamples = query_lims('Samples')[['Name', 'Parent_sample']]
        LIMSseqsamples = query_lims('Seqsamples')[['Sequencing_sample', 'Sample_Name']]
        parent_samples = pd.merge(
            LIMSseqsamples, 
            LIMSsamples, 
            left_on='Sample_Name', 
            right_on="Name", 
            how="left"
        ).drop_duplicates()
    
        parent_sample = parent_samples.loc[parent_samples['Sample_Name']==sample]['Parent_sample'].iloc[0]
        
        return parent_sample

    mutation_df = df.copy()

    def get_excel_cols_for_sample(excel_col_name_dict, seqsamples):
        excel_col_names = sorted([excel_col_name_dict[x] for x in seqsamples])
        first_col = excel_col_names[0]
        last_col = excel_col_names[-1]
        sheet_name = f"{first_col} to {last_col}"
        return sheet_name
        
    df_col_names = df.reset_index().columns
    excel_col_names = [get_column_letter(i) for i in range(1, len(df_col_names)+1)]
    excel_col_name_dict = dict(zip(
        df_col_names, excel_col_names
    ))
    
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    
        ## First sheet contains all columns
        mutation_df.reset_index().to_excel(writer, sheet_name="all_samples", index=False)
    
        ## Subsequent sheets only show index cols + parent + all transfers of one sample
        ## Excel sheet names can only have 31 chars, which is much less than many of the seqsample names.
        ## Therefore, name sheets according to Excel column names (A, B, C...AA, AB, ...)
        
        for name, group in breseq_summary.groupby('sample', sort=False):
            if name=='NA':
                continue
            try:
                parent_sample = get_parent_sample(name)
                cols = [parent_sample] + group['seqsample'].to_list()
            except:
                cols = group['seqsample'].to_list()

            trimmed_df = mutation_df[cols]#.copy()
            trimmed_df = trimmed_df.loc[~(trimmed_df==0).all(axis=1)] # Discard mutations that are zero in all samples          
            sheet_name = get_excel_cols_for_sample(excel_col_name_dict, group['seqsample'].to_list())
            trimmed_df.reset_index().to_excel(writer, sheet_name=sheet_name, index=False)
    

# ------ Format workbook: format header, format columns ------ 

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

def format_mutation_comparison_wb(excel_path, highlight_colors):
    
    wb = load_workbook(excel_path)
    
    ## Border formatting ----------
    thin = Side(style="thin")
    
    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                cell.border = border
    
    for s in wb.sheetnames:
        ws = wb[s]
        header_row = ws[1]
        
        ## Header row formatting ----------
        header_font = Font(bold=True)
        header_alignment = Alignment(wrap_text=True)
        for cell in header_row:
            cell.font = header_font
            cell.alignment = header_alignment
            ws.freeze_panes = "A2"
        
        ## Highlight columns ----------
        col_names = {}
        for i, cell in enumerate(header_row):
            col_names[cell.value] = i
    
        for key, value in col_names.items():
            target_column_index = col_names[key]
            
            # Iterate through the rows and access the cell in the target column
            for row_cells in ws.iter_rows(min_row=1): # Start from the first row
                cell = row_cells[target_column_index]
                cell.fill = highlight_colors[key]
    
    wb.save(excel_path)


def write_and_format_mutation_comparison_excel(excel_path,
                                               mutation_comparison_df,
                                               index_col_names,
                                               breseq_summary):

    print("Assigning fill colors to columns.")
    highlight_colors = create_highlight_color_dict(index_col_names, breseq_summary)
    print("\tDone.")
    print("Writing mutation comparison to Excel workbook.")
    write_mutation_comparison_wb(mutation_comparison_df, breseq_summary, excel_path)
    print("\tDone.")
    print("Formatting Excel workbook.")
    format_mutation_comparison_wb(excel_path, highlight_colors)
    print("\tDone.")

In [808]:
breseq_summary.head()

,seqsample,sample,strain,media,construct,replicate,transfer,is_parent,error,input_read_count,used_read_count,mapped_read_count,consensus_mutation_count,polymorphism_mutation_count,average_cov,ver_cassette,verAB,ver_cassette_CN,verAB_CN
171,ANL.stock.ACN3500.colony2,NA,NA,NA,NA,NA,NA,True,None,138419714,137954457,135137215,1,485,5254.4120,3851.130,4142.130,0.732933,0.788315
85,TFMN2.ACN3500.Iva.DEL6kb.2.T7.P,TFMN2.ACN3500.Iva.DEL6kb.2,ACN3500,Iva,DEL6kb,2,7,False,None,4102234,4089887,4006051,2,228,165.4533,137.498,146.269,0.831038,0.884050
109,TFMN2.ACN3500.Iva.DEL6kb.2.T13.P,TFMN2.ACN3500.Iva.DEL6kb.2,ACN3500,Iva,DEL6kb,2,13,False,None,3583178,3574138,3502026,2,237,145.0678,116.697,124.379,0.804431,0.857385
136,TFMN2.ACN3500.Iva.DEL6kb.2.T20.P,TFMN2.ACN3500.Iva.DEL6kb.2,ACN3500,Iva,DEL6kb,2,20,False,None,24634894,24569160,24086309,2,300,979.0606,862.780,900.508,0.881232,0.919767
32,TFMN2.ACN3500.Iva.DEL6kb.3.T2.P,TFMN2.ACN3500.Iva.DEL6kb.3,ACN3500,Iva,DEL6kb,3,2,False,None,6128818,6113116,5989700,1,262,246.1859,198.836,217.025,0.807666,0.881549


In [837]:
# Write all three versions

cutoffs = {
    "": df,
    "_greater5": df.loc[(df > 0.05).any(axis=1)].dropna(how="all"),
    "_greater80": df.loc[(df > 0.80).any(axis=1)].dropna(how="all")
}

for suffix, dataframe in cutoffs.items():
    
    excel_path = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted{suffix}.xlsx')
    
    write_and_format_mutation_comparison_excel(excel_path,
                                           dataframe,
                                           index,
                                           breseq_summary)

Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.
Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.
Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.


In [849]:
# Copying parent sample bam into jupyter environment for download and viz in IGV

parent_3500_bam = Breseq.from_existing(parent_3500.breseq / 'breseq_0b03256c7a').bam_path
parent_3500_bai = str(parent_3500_bam) + ".bai"
print(parent_3500_bam)
print(parent_3500_bai)

for f in [parent_3500_bam, parent_3500_bai]:
    shutil.copy(f, "/storage/nspahr/tmp/")

/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN3500.colony2/breseq_0b03256c7a/data/reference.bam
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN3500.colony2/breseq_0b03256c7a/data/reference.bam.bai


## Breseq analysis TFMN3

(Much of the following code should probably have its own module.)

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate comparison mutation table for all samples.

In [760]:
exp = 'TFMN3'

In [761]:
seqsamples = []
seqsamples.append(parent_3575)
seqsamples.append(parent_3749)
ALEseqsamples = [SeqSample(short, row['Sequencing_sample']) for _, row in LIMSseqsamples.iterrows()]
ALEseqsamples = [s for s in ALEseqsamples if exp in s.sample_name]
seqsamples += ALEseqsamples

In [460]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [762]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN3\.(ACN3500|ACN3575|ACN3749)\.(Mxb)\.(noDNA)\.([2-5]))\.T(\d{1,2})\.P$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        strain = str(match.group(2))
        media = str(match.group(3))
        construct = str(match.group(4))
        replicate = str(match.group(5))
        transfer = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        strain = 'NA'
        media = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'

    return {'sample': sample, 'strain': strain, 'media': media, 'construct': construct, 'replicate': replicate, 'transfer': transfer}
        

In [763]:
from collections import Counter
from aisynbiopipeline.workflows.breseq import Breseq
from aisynbiopipeline.workflows.mapping import PileupCol
import numpy as np

def create_breseq_summary(seqsample_batch, version_name, output_path=None, regions=None, loci=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        
        if loci:
            for key, value in loci['mutations'].items():
                prefix = key
                basecalls = PileupCol(b.bam_path, value['locus']).basecalls
                locus_cov = len(basecalls)
                counts = Counter(basecalls)
                try:
                    alt_freq = counts[value['alt_allele']] / locus_cov
                except ZeroDivisionError:
                    alt_freq = None
                row[key+'_locus_cov'] = locus_cov
                row[key+'_alleleCounts'] = dict(counts)
                row[key+'_alt_allele_freq'] = alt_freq
        
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    breseq_summary['is_parent'] = breseq_summary['seqsample'].apply(lambda x: x.startswith('ANL'))
    
    # Put rows and cols into correct order
    metadata_cols = ['seqsample', 'sample', 'strain', 'media', 'construct', 'replicate', 'transfer', 'is_parent']
    other_cols = [col for col in breseq_summary.columns.to_list() if col not in metadata_cols]
    breseq_summary = breseq_summary[metadata_cols + other_cols]
    breseq_summary.sort_values(
        ['is_parent', 'media', 'construct', 'strain', 'replicate', 'transfer'],
        ascending=[False, True, True, True, True, True],
        inplace=True
    )

    # Write breseq run summary to csv
    if output_path:
        breseq_summary.to_csv(output_path, index=False)
    
    return breseq_summary

In [463]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB')
}

In [ ]:
# loci = {
#     'reference': 'ACN3500_NSS',
#     'mutations': {
#         'promoter_6kb.DEL': {
#             'locus': (950040, 955945),  # added 50 bp from start and subtracted 50 bp from end to accomodate slightly different start and end loci between samples
#             'ref_allele': '...',
#             'alt_allele': ''
#         },
#         'vanK_1-bp.DEL': {
#             'locus': (974618, 974618),
#             'ref_allele': 'G',
#             'alt_allele': ''
#         },
#         'ACIAD_RS08195<->fecI.SNP': {
#             'locus': (1405240, 1405241),  ### THIS IS WRONG!!!
#             'ref_allele': 'T',
#             'alt_allele': 'A'
#         },
#         'dsbD.Q75stop': {
#             'locus': (3465740, 3465740),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'iscR_2-bp.DEL': {
#             'locus': (1405240, 1405241),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'rpoD_3-bp.DEL': {
#             'locus': (2860637, 2860639),
#             'ref_allele': 'GAA',
#             'alt_allele': ''
#         },
#         'adeK_2-bp.DEL': {
#             'locus': (2883783, 2883784),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'mnmA.R245H': {
#             'locus': (1230459, 1230459),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'ACIAD_RS01630_2-bp.DEL': {
#             'locus': (345201, 345202),
#             'ref_allele': 'AA',
#             'alt_allele': ''
#         }
#     }
# }

In [764]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

breseq_folder = home_dir + '/' + exp + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

breseq_summary = create_breseq_summary(
    seqsamples,
    breseq_version_name,
    output_path=os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_summary.csv'),
    regions=regions_sub,
    # loci=loci
)

# create_html_comparison(
#     seqsample_batch,
#     breseq_version_name,
#     os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.html')
# )

In [765]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

breseq_objects = []
for s in seqsamples:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ breseq_version_name
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

reference = 'ACN3500_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

breseq_outfolder = home_dir + '/' + exp + '/' + breseq_version_name

table_format = 'html'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

In [766]:
compare_df = pd.read_csv(csv)

In [767]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Need to exclude these because they will mess with pivoting on title
cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies',
    ]

# Exclude these because they are not of interest
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment', 'transl_table']

cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

# Columns that should only have integer values. Some may have weird text which messes with conversion to nullable int type.
# Force to numeric
int_cols = [
    'aa_position',
    'codon_number',
    'codon_position',
    'gene_position',
    'insert_position',
    'position',
    'position_end',
    'position_start',
    'repeat_length',
    'size',
]
for col in int_cols:
    compare_df[col] = pd.to_numeric(compare_df[col], errors='coerce')

# Now that expected numeric cols are numeric, can convert to int if possible
compare_df = compare_df[cols_to_keep].convert_dtypes()

# Define columns that will form the Multiindex 
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

# Create the pivoted mutation frame, where distinct mutations are rows, samples are columns, values are frequencies
df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)

# Order the comparison just like the summary
# df = df.loc[~(df > 0).all(axis=1)] # Only mutations that don't appear in all samples
df = df[breseq_summary['seqsample'].to_list()]

In [477]:
# OLD

# # Write reformatted mutations to csv.
# df.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted.csv'))
# df_greater5.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater5.csv'))
# df_greater80.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater80.csv'))

In [481]:
# OLD

# df.reset_index().to_excel(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted.xlsx'))

In [768]:
# Write all three versions

cutoffs = {
    "": df,
    "_greater5": df.loc[(df > 0.05).any(axis=1)].dropna(how="all"),
    "_greater80": df.loc[(df > 0.80).any(axis=1)].dropna(how="all")
}

for suffix, dataframe in cutoffs.items():
    
    excel_path = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted{suffix}.xlsx')
    
    write_and_format_mutation_comparison_excel(excel_path,
                                           dataframe,
                                           index,
                                           breseq_summary)

Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.
Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.
Assigning fill colors to columns.
	Done.
Writing mutation comparison to Excel workbook.
	Done.
Formatting Excel workbook.
	Done.


# Resequenced samples

Email to Plasmidsaurus:

Hi Olga,

As Nidhi indicated in an earlier message, I am reaching out because of read count issues in our most recent sequencing order, GFHFWC.

In this order, there were 262 samples on 3 plates. Of these samples, 64 have less than 2.4M reads, which is the number of reads required to get ~100x coverage over our genome. Several samples have nearly no reads at all. This is surprising because OD measurements taken prior to shipping showed that all samples should have had comparable cell counts and more than enough DNA for sequencing. We expected read counts for the present order to be similar to read counts in our last custom Illumina order from December '25 (MQRBN8), which had on average 19M reads per sample. (Please find the MultiQC reports for each of these orders attached.)

Of the 3 plates in order GFHFWC, only plates 1 and 2 seem to be affected by low read counts:

Plate 1:     4.2M reads on average
Plate 2:    4.4M reads on average
Plate 3:    14.3M reads on average

I also noticed that on plate 1, read numbers for all samples in the first column (A1-H1) were especially poor (see attached excel file, GFHFWC_read_counts.xlsx).

Thank you for looking into this,

Natascha

We had all samples with less than 100X coverage resequenced (Seqorder YWLYRK). Of these, 4 samples failed yet again. They were resequenced in Seqorder WTN32D.

Plan:
1. Download the 2 seqorders.
2. Import re-run sample names. (was easiest to do this by hand)
3. Rename seqsamples in the two seqorders.
4. Create batch of seqsamples in these two seqorders.
5. QA/QC on batch.
6. Breseq on batch.
7. Re-run breseq analysis.
8. Upload analysis files to LIMS and share.

## Download seqorders

In [4]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_codes = ['YWLYRK', 'WTN32D']

for item_code in item_codes:
    seqorder_name = create_seqorder_name(item_code)
    
    reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
    os.makedirs(reception_dir, exist_ok=True)
    print(reception_dir)

/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-24_YWLYRK
/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-04-13_WTN32D


In [5]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)

for item_code in item_codes:
    seqorder_name = create_seqorder_name(item_code)
    reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
    download_results(item_code, access_token, reception_dir)

ITEM YWLYRK
{'code': 'YWLYRK',
 'done_date': '2026-03-24T21:48:12.996882+00:00',
 'gross': 2310.0,
 'order_name': 'HANKE_260312_reruns_of_GFHFWC',
 'product_name': 'custom_illumina',
 'quantity': 66,
 'status': 'complete'}



DOWNLOADING RESULTS FOR YWLYRK 

No results found for YWLYRK: b'{"message": "Error getting item results link"}\n'
DOWNLOADING READS FOR YWLYRK 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-24_YWLYRK/YWLYRK_reads.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-24_YWLYRK/YWLYRK_reads.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-24_YWLYRK/YWLYRK_reads
ITEM WTN32D
{'code': 'WTN32D',
 'done_date': '2026-04-13T17:53:19.690577+00:00',
 'gross': 4.0,
 'order_name': 'rerun_YWLYRK',
 'product_name': 'custom_illumina',
 'quantity': 4,
 'status': 'complete'}



DOWNLOADING RESULTS FOR WTN32D 

No results found for WTN32D

In [6]:
# Ensure that you unzipped the expected number of fastq files

for item_code in item_codes:
    seqorder_name = create_seqorder_name(item_code)
    reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
    plasmidsaurus_read_folder_name = os.path.join(reception_dir, item_code + '_reads')
    n_fastqfiles = len([x for x in os.listdir(plasmidsaurus_read_folder_name) if x.endswith(".fastq.gz")])
    print(seqorder_name)
    print(n_fastqfiles)

Plasmidsaurus_2026-03-24_YWLYRK
132
Plasmidsaurus_2026-04-13_WTN32D
8


## Create seqorders and rename files

In [19]:
item_code = 'GFHFWC'
seqorder_name = create_seqorder_name(item_code)
print(seqorder_name)
# reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
# plasmidsaurus_read_folder_name = os.path.join(reception_dir, item_code + '_reads')
# os.listdir(plasmidsaurus_read_folder_name)

Plasmidsaurus_2026-03-04_GFHFWC


In [16]:
from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.fastq_utils import create_manifest, parse_illumina_fastq_filename

# The two seqorders have different file name formats :-(
item_code = 'YWLYRK'
seqorder_name = create_seqorder_name(item_code)
reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
plasmidsaurus_read_folder_name = os.path.join(reception_dir, item_code + '_reads')
print(plasmidsaurus_read_folder_name)
manifest = create_manifest(plasmidsaurus_read_folder_name, platform='illumina')
display(manifest)
for s in manifest['sample_name']:
    print(s)

item_code = 'WTN32D'
seqorder_name = create_seqorder_name(item_code)
reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
plasmidsaurus_read_folder_name = os.path.join(reception_dir, item_code + '_reads')
print(plasmidsaurus_read_folder_name)
manifest = create_manifest(plasmidsaurus_read_folder_name, platform='plasmidsaurus_illumina')
display(manifest)
for s in manifest['sample_name']:
    print(s)

/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-03-24_YWLYRK/YWLYRK_reads


,sample_name,sample_number,fwd_fastq,rvs_fastq
0,YWLYRK_1,94,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,YWLYRK_10,103,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,YWLYRK_11,104,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,YWLYRK_12,105,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,YWLYRK_13,106,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
...,...,...,...,...
61,YWLYRK_65,158,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
62,YWLYRK_66,159,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
63,YWLYRK_7,100,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
64,YWLYRK_8,101,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


YWLYRK_1
YWLYRK_10
YWLYRK_11
YWLYRK_12
YWLYRK_13
YWLYRK_14
YWLYRK_15
YWLYRK_16
YWLYRK_17
YWLYRK_18
YWLYRK_19
YWLYRK_2
YWLYRK_20
YWLYRK_21
YWLYRK_22
YWLYRK_23
YWLYRK_24
YWLYRK_25
YWLYRK_26
YWLYRK_27
YWLYRK_28
YWLYRK_29
YWLYRK_3
YWLYRK_30
YWLYRK_31
YWLYRK_32
YWLYRK_33
YWLYRK_34
YWLYRK_35
YWLYRK_36
YWLYRK_37
YWLYRK_38
YWLYRK_39
YWLYRK_4
YWLYRK_40
YWLYRK_41
YWLYRK_42
YWLYRK_43
YWLYRK_44
YWLYRK_45
YWLYRK_46
YWLYRK_47
YWLYRK_48
YWLYRK_49
YWLYRK_5
YWLYRK_50
YWLYRK_51
YWLYRK_52
YWLYRK_53
YWLYRK_54
YWLYRK_55
YWLYRK_56
YWLYRK_57
YWLYRK_58
YWLYRK_59
YWLYRK_6
YWLYRK_60
YWLYRK_61
YWLYRK_62
YWLYRK_63
YWLYRK_64
YWLYRK_65
YWLYRK_66
YWLYRK_7
YWLYRK_8
YWLYRK_9
/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-04-13_WTN32D/WTN32D_reads


,sample_name,fwd_fastq,rvs_fastq
0,WTN32D_1_rerun_GFHFWC_26_10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,WTN32D_2_rerun_GFHFWC_37_12,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,WTN32D_3_rerun_GFHFWC_38_13,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,WTN32D_4_rerun_GFHFWC_80_21,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


WTN32D_1_rerun_GFHFWC_26_10
WTN32D_2_rerun_GFHFWC_37_12
WTN32D_3_rerun_GFHFWC_38_13
WTN32D_4_rerun_GFHFWC_80_21


In [19]:
# Plasmidsaurus provides a sort of sample manifest for download from the seqorder page. (not available with read download through API).
# Downloaded to my laptop, uploaded to home_dir, now copying to reception dir.

import shutil

home_dir = '/storage/nspahr/lib_analysis/Plasmidsaurus_2026-03-04_GFHFWC'

for item_code in item_codes:
    seqorder_name = create_seqorder_name(item_code)
    reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
    shutil.copy2(os.path.join(home_dir, f'{item_code}-summary-report.csv'), reception_dir)

## Rename sample names

In [17]:
import pandas as pd

home_dir = '/storage/nspahr/lib_analysis/Plasmidsaurus_2026-03-04_GFHFWC'

rerun_samples_df = pd.read_csv(
    os.path.join(home_dir, 'rerun_sample_names.csv')
)
rerun_samples_df

,GFHFWC_name,YWLYRK_sample_name,YWLYRK_file_name,WTN32D_sample_name,WTN32D_file_name
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,GFHFWC_1,YWLYRK_1,NaN,NaN
1,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,GFHFWC_2,YWLYRK_2,NaN,NaN
2,TFMN2.ACN3500.Iva.DEL6kb.3.T1.P,GFHFWC_10,YWLYRK_3,NaN,NaN
3,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.3.T1.P,GFHFWC_11,YWLYRK_4,NaN,NaN
4,TFMN2.ACN3500.Iva.gDNA3560.3.T1.P,GFHFWC_13,YWLYRK_5,NaN,NaN
...,...,...,...,...,...
61,TFMN2.ACN3500.Van.DEL6kb-DELvanK.2.T13.P,GFHFWC_183,YWLYRK_62,NaN,NaN
62,TFMN2.ACN3500.Iva.DEL6kb.4.T13.P,GFHFWC_184,YWLYRK_63,NaN,NaN
63,TFMN2.ACN3500.Iva.gDNA3560.4.T13.P,GFHFWC_187,YWLYRK_64,NaN,NaN
64,TFMN2.ACN3500.Iva.gDNA3575.4.T13.P,GFHFWC_188,YWLYRK_65,NaN,NaN


In [33]:
# YWLYRK: Copy fastqs into library folder

from pathlib import Path
import shutil
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

item_code = 'YWLYRK'
seqorder_name = create_seqorder_name(item_code)
reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
manifest = create_manifest(reads_path, platform='illumina')
# seqorder = SeqOrder(seqorder_name, create=True)
# short = Library(seqorder, 'Illumina', create=True)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['fwd_fastq'])
    plasmidsaurus_samplename = "_".join(plasmidsaurus_basename.split("_")[:2])
    # print(plasmidsaurus_samplename)
    aisynbio_samplename = rerun_samples_df.loc[rerun_samples_df['YWLYRK_file_name']==plasmidsaurus_samplename]['GFHFWC_name'].iloc[0]
    # print(aisynbio_samplename)
    aisynbio_basename = aisynbio_samplename + "_R1.fastq.gz"
    # print(aisynbio_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['rvs_fastq'])
    plasmidsaurus_samplename = "_".join(plasmidsaurus_basename.split("_")[:2])
    # print(plasmidsaurus_samplename)
    aisynbio_samplename = rerun_samples_df.loc[rerun_samples_df['YWLYRK_file_name']==plasmidsaurus_samplename]['GFHFWC_name'].iloc[0]
    # print(aisynbio_samplename)
    aisynbio_basename = aisynbio_samplename + "_R2.fastq.gz"
    # print(aisynbio_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

In [40]:
# WTN32D: Copy fastqs into library folder

from pathlib import Path
import shutil
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

item_code = 'WTN32D'
seqorder_name = create_seqorder_name(item_code)
reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
manifest = create_manifest(reads_path, platform='plasmidsaurus_illumina')
# seqorder = SeqOrder(seqorder_name, create=True)
# short = Library(seqorder, 'Illumina', create=True)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['fwd_fastq'])
    plasmidsaurus_samplename = "_".join(plasmidsaurus_basename.split("_")[:6])
    # print(plasmidsaurus_samplename)
    aisynbio_samplename = rerun_samples_df.loc[rerun_samples_df['WTN32D_file_name']==plasmidsaurus_samplename]['GFHFWC_name'].iloc[0]
    # print(aisynbio_samplename)
    aisynbio_basename = aisynbio_samplename + "_R1.fastq.gz"
    # print(aisynbio_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['rvs_fastq'])
    plasmidsaurus_samplename = "_".join(plasmidsaurus_basename.split("_")[:6])
    # print(plasmidsaurus_samplename)
    aisynbio_samplename = rerun_samples_df.loc[rerun_samples_df['WTN32D_file_name']==plasmidsaurus_samplename]['GFHFWC_name'].iloc[0]
    # print(aisynbio_samplename)
    aisynbio_basename = aisynbio_samplename + "_R2.fastq.gz"
    # print(aisynbio_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

## Create batch of seqsamples in these two seqorders

In [41]:
short = Library('Plasmidsaurus_2026-03-24_YWLYRK', platform="Illumina")
short.create_manifest("received")

,sample_name,R1,R2
0,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T2.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.3.T7.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.4.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
...,...,...,...
61,TFMN2.ACN3500.Van.gDNA3575.2.T1.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
62,TFMN2.ACN3500.Van.gDNA3749.1.T3.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
63,TFMN2.ACN3500.Van.gDNA3749.1.T7.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
64,TFMN2.ACN3500.Van.gDNA3749.2.T20.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [44]:
WTN32D_short = Library('Plasmidsaurus_2026-04-13_WTN32D', platform="Illumina")
WTN32D_short_manifest = WTN32D_short.create_manifest("received")
WTN32D_seqsamples = [SeqSample(WTN32D_short, sample) for sample in WTN32D_short_manifest['sample_name']]

In [45]:
YWLYRK_short = Library('Plasmidsaurus_2026-03-24_YWLYRK', platform="Illumina") #
YWLYRK_short_manifest = YWLYRK_short.create_manifest("received")
YWLYRK_seqsamples = [SeqSample(YWLYRK_short, sample) for sample in YWLYRK_short_manifest['sample_name']]

## QA/QC

In [ ]:
## To run fastp celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
cd ~/code/AISynbioPipeline/
python -m aisynbiopipeline.tasks.fastp_task 1
"""

In [ ]:
## fastp workers running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 2

In [46]:
# Create the trimmed subfolder
WTN32D_short.create_subfolder('trimmed')
YWLYRK_short.create_subfolder('trimmed')

In [56]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    received_dir = fwd_in_path.parent
    library_dir = received_dir.parent
    fwd_out_path = os.path.join(library_dir, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library_dir, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [57]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in WTN32D_seqsamples + YWLYRK_seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(sample),
        queue='fastp'
    )
    results.append(result)

In [62]:
for i in results:
    print(i.status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS
STARTED
STARTED
STARTED
STARTED
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING


In [63]:
all([(r.status=='SUCCESS') for r in results])

True

In [64]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

item_codes = ['YWLYRK', 'WTN32D']

for item_code in item_codes:
    seqorder_name = create_seqorder_name(item_code)
    short = Library(seqorder_name, platform="Illumina")
    multiqc_report = run_multiqc(short.path / 'trimmed')
    # multiqc_report_dir = os.path.dirname(multiqc_report)
    multiqc_report_file = os.path.basename(multiqc_report)
    dst_multiqc_report_file = os.path.join(home_dir, item_code + '_trimmed_' + multiqc_report_file)
    shutil.copy(multiqc_report, dst_multiqc_report_file)


/// ]8;id=307517;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.34 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-03-24_YWLYRK/Plasmidsaurus_2026-03-24_YWLYRK_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 264/264                                                   html

             fastp | Found 66 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete



/// ]8;id=63363;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.34 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-04-13_WTN32D/Plasmidsaurus_2026-04-13_WTN32D_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 16/16                                                   html

             fastp | Found 4 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


## Run breseq

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
cd ~/code/AISynbioPipeline/
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# four breseq workers are running!

In [65]:
# Create breseq dirs

WTN32D_short.create_subfolder('breseq')
YWLYRK_short.create_subfolder('breseq')

In [66]:
# Where should this code go?

# Specifies and assigns breseq parameters

from pathlib import Path

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4, polymorphism_frequency_cutoff=0.05):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors,
        'polymorphism_frequency_cutoff': polymorphism_frequency_cutoff
        
    }
    return breseq_params

In [67]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

results = []

for sample in WTN32D_seqsamples + YWLYRK_seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk', polymorphism_frequency_cutoff=0.005, fold_coverage=0),
        queue='breseq'
    )
    results.append(result)    

In [68]:
sum([(r.status=='SUCCESS') for r in results])

0

In [69]:
for i in results:
    print(i.status)
for i in results:
    print(i.result['output']) ## Check version_name for breseq run

STARTED
STARTED
STARTED
STARTED
STARTED
STARTED
STARTED
STARTED
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING
PENDING


KeyError: 'output'

## Breseq analysis